# Regulus — the cited crosswalk graph (Phase 2)

Build the **regulatory knowledge graph** over the ingested provisions and a
curated, **cited** crosswalk table, then look up an issue and see its applicable
provisions annotated with the **risks** they address and their **cross-framework
references** — each with a citation.

`CROSSWALK` edges come only from `data/crosswalks/crosswalks.csv` (never inferred);
`ADDRESSES` risk tags are keyword-derived and marked low-confidence.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))
try:
    import geometric_knowledge_network  # noqa: F401
except ModuleNotFoundError:
    gkn_src = Path.cwd().parent.parent / 'geometric_knowledge_network' / 'src'
    if gkn_src.exists():
        sys.path.insert(0, str(gkn_src))
        print('Using local GKN checkout at', gkn_src)
    else:
        raise ModuleNotFoundError("Install GKN: pip install git+https://github.com/minw0607/geometric_knowledge_network")

from regulus.config import RegulusConfig
from regulus.standards_loader import StandardsLoader
from regulus.crosswalk import load_crosswalks
from regulus.graph import RegulusGraphBuilder, graph_summary
from regulus.graph_lookup import RegulusGraphLookup

config = RegulusConfig()
provisions = StandardsLoader(config).load(framework_ids=['eu_ai_act', 'nist_ai_rmf'])
crosswalks = load_crosswalks()
print('provisions:', len(provisions), '| crosswalk rows:', len(crosswalks))

## Build the graph

In [ ]:
graph = RegulusGraphBuilder().build(provisions, crosswalks)
for key, count in graph_summary(graph).items():
    print(f'  {key:24s} {count}')

## Crosswalk-aware lookup

Each applicable provision is annotated with the risks it addresses and its cited
cross-framework references.

In [ ]:
gl = RegulusGraphLookup(provisions, config)

for issue in [
    'Our credit model was deployed without testing for demographic bias.',
    'We have no post-deployment monitoring for our high-risk AI system.',
]:
    print('=' * 100)
    print('ISSUE:', issue, '\n')
    for r in gl.search(issue, top_k=3):
        print(f'* {r.provision.citation()}   (score {r.score:.3f})')
        if r.risks:
            print(f'    risks: {", ".join(r.risks)}')
        for cx in r.crosswalks:
            print(f'    -> {cx.provision.citation()}  [{cx.relation}]')
            print(f'        rationale: {cx.rationale}')
            print(f'        source: {cx.source}')
    print()

## Next: Phase 3

Use GKN's multi-hop retriever and path explainer to return the full **evidence
path** (issue → provision → crosswalk → provision) and expand along the graph,
then add an LLM interpretation layer that produces a structured, cited answer.

To improve/extend the crosswalks, edit `data/crosswalks/crosswalks.csv` — replace
the seed rows with authoritative mappings and add more frameworks.